In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os
import re

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

Allow changes to imported Python files without reseting the kernel

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_processed
%store -r sales_data_processed

Import data

In [ ]:
# Check if the data is already imported
if 'static_data_processed' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"palate_data_parquet")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"palate_data_parquet/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_processed = static_data.copy()
    %store static_data_processed


# Data already exists
else:
    static_data = static_data_processed.copy()

# Check if the data is already imported
if 'sales_data_processed' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"palate_data_parquet/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"palate_data_parquet/orders_item_level/{filename}")
            location_id = re.sub(r'\.parquet$', '', filename)
            sales_data[location_id] = df
    
    # Rename and store
    sales_data_processed = sales_data.copy()
    %store sales_data_processed

# Data already exists
else:
    sales_data = sales_data_processed.copy()

Retrieve the four static reference data frames

In [ ]:
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_data.keys()) # List out locations

Merge menu and sales

In [ ]:
items_tagged.drop_duplicates(subset=['item_name', 'location_id'], inplace=True)

# Check if 'items_tagged' has unique pairs of 'item_name' and 'location_id'
if items_tagged.duplicated(subset=['item_name', 'location_id']).any():
    raise ValueError("Duplicates found in 'items_tagged' for the combination of 'item_name' and 'location_id'")

merged_sales_and_menu = {}
merged_rights = {}
for location_id, df in tqdm(sales_data.items()):

    # Copy to prevent overwriting
    df_copy = df.copy()
    items_tagged_copy = items_tagged.copy()

    df_copy.dropna(how='all', inplace=True)
    df_copy.dropna(subset=['item_name'], inplace=True)
    items_tagged_copy.dropna(how='all', inplace=True)
    items_tagged_copy.dropna(subset=['item_name'], inplace=True)

    # For ease of merging, pretend items of different capitalizations are the same
    df_copy['item_name'] = df_copy['item_name'].str.lower()
    items_tagged_copy['item_name'] = items_tagged_copy['item_name'].str.lower()
    items_tagged_copy.drop_duplicates(['item_name', 'location_id'], inplace=True) 
    
    # Perform the merge on both 'item_name' and 'location_id'
    merged = pd.merge(df_copy.reset_index(), items_tagged_copy, 
                      on=['item_name', 'location_id'], how='left')
    
    temp = items_tagged_copy[items_tagged_copy['location_id'] == location_id].copy()

    # Find the merge miss item name versions from the menu data
    merged_right = pd.merge(df_copy.reset_index(), temp, 
                      on=['item_name', 'location_id'], how='right')

    # Don't remove failed merges

    # Set 'created_at' back as the index
    merged.set_index('created_at', inplace=True)
    merged_right.set_index('created_at', inplace=True)

    # Recapitalize
    merged['item_name'] = merged['item_name'].str.title()
    merged_right['item_name'] = merged_right['item_name'].str.title()

    # Store in the dictionary
    merged_sales_and_menu[location_id] = merged
    merged_rights[location_id] = merged_right

Merge misses

In [ ]:
total_merge_misses = -30
for i in range(30):
    merge_miss = merged_sales_and_menu[location_ids[i]][merged_sales_and_menu[location_ids[i]]['is_plant_based'].isna()]['item_name'].unique().size
    total_merge_misses += merge_miss

Examples

In [ ]:
# Example of problem 1, missing menu data for 27
fdf(items_tagged).multi_filter([('location_id', location_ids[27])])#['item_name'].str.lower().sort_values().iloc[50:].head(20)

# Example of problem 2, encoding error for multiple restaurants
fdf(items_tagged).multi_filter([('location_id', location_ids[2])])['item_name'].str.lower().sort_values().iloc[622:].head(20)
# fdf(items_tagged).filter('item_name', 'kahl√∫a') # <----- doesn't have it like it should

Aggregate encoding errors

In [ ]:
encoding_errors = {}
all_encoding_errors = set()
for i, loc_id in enumerate(location_ids):
    if i != 27:
        df = merged_sales_and_menu[loc_id].copy()
        df.drop_duplicates('item_name', inplace=True)
        df = df[~df['item_name'].isna() & df['is_plant_based'].isna()]
        df = df[~df['item_name'].str.contains('nan')]
        if loc_id not in encoding_errors:
            encoding_errors[loc_id] = set()
        if not df['item_name'].empty:
            encoding_errors[loc_id] = encoding_errors[loc_id].union(set(df['item_name'].values.tolist()))
            all_encoding_errors = all_encoding_errors.union(set(df['item_name'].values.tolist()))

Merge miss encoding errors

In [ ]:
right_encoding_errors = {}
right_all_encoding_errors = set()
for i, loc_id in enumerate(location_ids):
    if i != 27:
        df = merged_rights[loc_id].copy()
        df.drop_duplicates('item_name', inplace=True)
        df = df[~df['item_name'].isna() & df['order_id'].isna()]
        df = df[~df['item_name'].str.contains(' nan ')]
        if loc_id not in right_encoding_errors:
            right_encoding_errors[loc_id] = set()
        if not df['item_name'].empty:
            right_encoding_errors[loc_id] = right_encoding_errors[loc_id].union(set(df['item_name'].values.tolist()))
            right_all_encoding_errors = right_all_encoding_errors.union(set(df['item_name'].values.tolist()))

Manually paired up items with encoding errors (main code cell)

In [ ]:
# The six restaurants with encoding errors respectively
ids_encoding_errors = ['3AXDVZJYN9DRS', 'C0BE4NDSW26QN', 'L3XS7WSJ4AJA3', 'LBMCPAYT7W36V', 'ED5J990H5VAZT', '2HRX9P6HKXA8V']

# '3AXDVZJYN9DRS'
res1_sales_to_remove = [
  '2023-01-09 00:00:00',
  '2023-03-09 00:00:00',
  '2023-03-10 00:00:00',
  '2023-03-12 00:00:00',
  '2023-05-11 00:00:00',
  '2023-06-11 00:00:00',
  '2023-10-12 00:00:00',
  '2023-11-12 00:00:00',
  '2023-12-09 00:00:00',
  '2023-12-10 00:00:00',
  '2023-12-12 00:00:00',
  'Regal Disco Ukraine ¬£10 Donation',
  'Regal Disco Ukraine ¬£5 Donation',
  'Ticket ¬£10',
  'Ticket ¬£10 Lz & Friends'
]

res1_sales_encodings = ['2. 5-7Pm ¬£6 Pizza',
  'Brewdogs, 3 For ¬£10',
  'Kahl√∫A',
]

res1_menu_encodings = ['2. 5-7Pm ¬¨¬£6 Pizza',
 'Brewdogs, 3 For ¬¨¬£10',
 'Kahl‚Àö‚À´A',
]

res1_menu_encodings_unused = [
  'Regal Disco Ukraine ¬¨¬£10 Donation',
 'Regal Disco Ukraine ¬¨¬£5 Donation',
 'Ticket ¬¨¬£10',
 'Ticket ¬¨¬£10 Lz & Friends'
]

# 'C0BE4NDSW26QN'
res2_sales_encodings = ['Binary Solo\uf8ffÜ§Ñ',
  'Blake‚Äôs Apple Lantern 12Oz',
  'Blake‚Äôs Grizzly Pear Cider 16',
  'Blake‚Äôs Rainbow Seeker 16Oz',
  'Crudit√©S',
  'Cuv√©E De Tomme 10Oz',
  'Cuv√©E Rouge 12Oz',
  'Jalape√±O Burger',
  'Jp Best Lei‚Äôd 10Oz',
  'Kat\uf8ffÜèåô∏È‚Äç‚Ôçô∏È 12Oz',
  'Kilbane‚Äôs',
  'Mol√© Stout10',
  'Os Dry Rose \uf8ffÜåπ',
  'Sg Bell‚Äôs Oberon',
  'Sg Cuv√©E De Tomme',
  'Sg Cuv√©E Rouge',
  'Sg Jp Best Lei‚Äôd',
  'Sg Kat\uf8ffÜèåô∏È‚Äç‚Ôäô∏È',
  'Sg Mol√©',
  'Sg Shacks Ros√©',
  'Sg Shacks Ros√®',
  'Sg \uf8ffÜåà Seeker',
  "Sh Kilbane'S Irish Stout \uf8ffÜçä",
  'Shacks Ros√© 16Oz',
  'Shacks Ros√®',
  'Shacks Ros√® 12Oz',
  'Wc Rackman‚Äôs Rye Bock 16Oz',
  '\uf8ffÜ§¢Lichtenhainer',
  '\uf8ffÜß∏Blake‚Äôs Grizzly Pear Can',
  '\uf8ffÜåà Seeker 16Oz.',
  '\uf8ffÜåù Gold',
  '\uf8ffÜåù Gold 16Oz',
  '\uf8ffÜåû Oberon 16Oz',
  '\uf8ffÜç™ Mini Cookie',
  '\uf8ffÜêì And Waffles Stout',
  '\uf8ffÜêì And Waffles Stout W/ Vanilla',
  '\uf8ffÜëã\uf8ffÜèø Goodbye Forever 12Oz',
  '\uf8ffÜëã\uf8ffÜèø Goodbye Forver',
  '\uf8ffÜí°Singlecut 18 Watt 16Oz',
  '\uf8ffÜíégem Cutter 12Oz',
  '\uf8ffÜî® King Of The Hammer 12Oz']

res2_menu_encodings = [
  'Binary Soloô£Ø√º¬Ss√±',
  'Blake‚Äö√Ñ√¥S Apple Lantern 12Oz',
  'Blake‚Äö√Ñ√¥S Grizzly Pear Cider 16',
  'Blake‚Äö√Ñ√¥S Rainbow Seeker 16Oz',
  'Crudit‚Àö¬©S',
  'Cuv‚Àö¬©E De Tomme 10Oz',
  'Cuv‚Àö¬©E Rouge 12Oz',
  'Jalape‚Àö¬±O Burger',
  'Jp Best Lei‚Äö√Ñ√¥D 10Oz',
  'Katô£Ø√º√®√•√Î‚Àè√®‚Äö√Ñ√Ss‚Äö√¥√Á√Î‚Àè√® 12Oz',
  'Kilbane‚Äö√Ñ√¥S',
  'Mol‚Àö¬© Stout10',
  'Os Dry Rose Ô£Ø√º√•Œä',
  'Sg Bell‚Äö√Ñ√¥S Oberon',
  'Sg Cuv‚Àö¬©E De Tomme',
  'Sg Cuv‚Àö¬©E Rouge',
  'Sg Jp Best Lei‚Äö√Ñ√¥D',
  'Sg Katô£Ø√º√®√•√Î‚Àè√®‚Äö√Ñ√Ss‚Äö√¥√Ñ√Î‚Àè√®',
  'Sg Mol‚Àö¬©',
  'Sg Shacks Ros‚Àö¬©',
  'Sg Shacks Ros‚Àö¬Æ',
  'Sg Ô£Ø√º√•√† Seeker',
  "Sh Kilbane'S Irish Stout Ô£Ø√º√Ss√Ñ",
 'Shacks Ros‚Àö¬© 16Oz',
 'Shacks Ros‚Àö¬Æ',
 'Shacks Ros‚Àö¬Æ 12Oz',
 'Wc Rackman‚Äö√Ñ√¥S Rye Bock 16Oz',
 'Ô£Ø√º¬Ss¬¢Lichtenhainer',
 'Ô£Ø√º√Ü‚Àèblake‚Äö√Ñ√¥S Grizzly Pear Can',
 'Ô£Ø√º√•√† Seeker 16Oz.',
 'Ô£Ø√º√•√Π Gold',
 'Ô£Ø√º√•√Π Gold 16Oz',
 'Ô£Ø√º√•√ª Oberon 16Oz',
 'Ô£Ø√º√Ss‚Ñ¢ Mini Cookie',
 'Ô£Ø√º√™√¨ And Waffles Stout',
 'Ô£Ø√º√™√¨ And Waffles Stout W/ Vanilla',
 'Ô£Ø√º√´√£Ô£Ø√º√®√∏ Goodbye Forever 12Oz',
 'Ô£Ø√º√´√£Ô£Ø√º√®√∏ Goodbye Forver',
 'Ô£Ø√º√≠¬∞Singlecut 18 Watt 16Oz',
 'Ô£Ø√º√≠√©Gem Cutter 12Oz',
 'Ô£Ø√º√Æ¬Æ King Of The Hammer 12Oz']


# 'L3XS7WSJ4AJA3'
res3_sales_encodings = [
  'Bailey‚Äôs On Ice 50Ml',
  'Brisket Chilli ‚Äòn‚Äô Chips',
  'Reggae Rice ‚Äòn‚Äô Beans',
  'Ribs ‚Äún‚Äù Wings',
  '¬£25 Little Tipsy Tea Ticket',
  '¬£35 Big Tipsy Tea Ticket'
]

res3_menu_encodings = [
  'Bailey‚Äö√Ñ√¥S On Ice 50Ml',
  'Brisket Chilli ‚Äö√Ñ√≤N‚Äö√Ñ√¥ Chips',
  'Reggae Rice ‚Äö√Ñ√≤N‚Äö√Ñ√¥ Beans',
  'Ribs ‚Äö√Ñ√∫N‚Äö√Ñ√Π Wings',
  '¬¨¬£25 Little Tipsy Tea Ticket',
  '¬¨¬£35 Big Tipsy Tea Ticket'
]

# 'LBMCPAYT7W36V'
res4_sales_to_label = [
  '*176 Bistro Burger',
  '*Applewood Smoked Blt',
  '*Buttermilk Fried Chicken Cutlet',
  '*Falafel Sliders',
  '*Rosemary Pecan Chicken Salad',
  '*Southern Standard',
  '*Zabar Deviled Egg Salad',
]

res4_sales_encodings = ['**Baked Cr√®Me Brulee French Toast',
  'Brown‚Äôs Court',
  'Brown‚Äôs Court Cookies',
  'Edmund‚Äôs Fruit Punch',
  'Edmund‚Äôs Oast - Sour Fruit Punch',
  'Farmer‚Äôs Omelette And Frites',
  'Glass - Albari√±O',
  'Glass - O Fillo Da Condessa Albari√±O',
  'John‚Äôs Island Tomato Pie',
  'Natalie‚Äôs 100% Grapefruit Juice',
  'Natalie‚Äôs Lemonade',
  'Natalie‚Äôs Oj',
  'Nueske‚Äôs Bacon',
  'O Fillo Da Condesa - 2021 Albari√±O, Spain',
  'Porter‚Äôs Petite',
  'Potter‚Äôs Sampler',
  'Rancher‚Äôs Steak Omelette',
  'Rosemary Sea Salt Fry Cup ‚Ìã',
  'Ros√© Lemonade Spritz',
  'San Pelegr√≠N‚Äôs',
  'Scuby‚Äôs Fund',
  'Summer Water - Bubbly Ros√©, France',
  'Tuesday Boar‚Äôs Head Turkey Sandwich',
  'Vegetarian - Whipped Hummus Plate ‚Ìã',
  'Villaviva - 2020 Vegan Ros√©, France',
  'Wendy‚Äôs Cookies',
  '‚Äúnot Arby‚Äôs‚Äù Roast Beef Sandwich']

res4_menu_encodings = [
  '**Baked Cr‚Àö¬Æme Brulee French Toast',
  'Brown‚Äö√Ñ√¥S Court',
  'Brown‚Äö√Ñ√¥S Court Cookies',
  'Edmund‚Äö√Ñ√¥S Fruit Punch',
 'Edmund‚Äö√Ñ√¥S Oast - Sour Fruit Punch',
 'Farmer‚Äö√Ñ√¥S Omelette And Frites',
 'Glass - Albari‚Àö¬±O',
 'Glass - O Fillo Da Condessa Albari‚Àö¬±O',
 'John‚Äö√Ñ√¥S Island Tomato Pie',
 'Natalie‚Äö√Ñ√¥S 100% Grapefruit Juice',
 'Natalie‚Äö√Ñ√¥S Lemonade',
 'Natalie‚Äö√Ñ√¥S Oj',
 'Nueske‚Äö√Ñ√¥S Bacon',
 'O Fillo Da Condesa - 2021 Albari‚Àö¬±O, Spain',
 'Porter‚Äö√Ñ√¥S Petite',
 'Potter‚Äö√Ñ√¥S Sampler',
 'Rancher‚Äö√Ñ√¥S Steak Omelette',
 'Rosemary Sea Salt Fry Cup ‚Äö√¨√£',
 'Ros‚Àö¬© Lemonade Spritz',
 'San Pelegr‚Àö‚Â†N‚Äö√Ñ√¥S',
 'Scuby‚Äö√Ñ√¥S Fund',
 'Summer Water - Bubbly Ros‚Àö¬©, France',
 'Tuesday Boar‚Äö√Ñ√¥S Head Turkey Sandwich',
 'Vegetarian - Whipped Hummus Plate ‚Äö√¨√£',
 'Villaviva - 2020 Vegan Ros‚Àö¬©, France',
 'Wendy‚Äö√Ñ√¥S Cookies',
 '‚Äö√Ñ√∫Not Arby‚Äö√Ñ√¥S‚Äö√Ñ√Π Roast Beef Sandwich'
]

# 'ED5J990H5VAZT'
res5_sales_to_remove = [
  'Book - Dreamer‚Äôs Journal',
  'Card - Emperor Penguin ‚Äúseasons Greetings‚Äù',
  'Card - Happy Mother‚Äôs Day',
  'Card - Herd It‚Äôs Somebody‚Äôs Bday',
  'Card - Ily Even When You‚Äôre Cranky',
  'Card - Omg You‚Äôre Old',
  'Card - Season‚Äôs Greetings',
  'Card - You‚Äôre A Great Mom',
  'Card - You‚Äôre A Mooooom',
  'Card - You‚Äôre The Best Dad',
  'Card - You‚Äôre The Tits',
  'Planner - Get ‚Äòer Done',
  'Postcard - Pdx St. John‚Äôs Bridge'                   
]

res5_sales_encodings = [
  'Caf√© Au Lait',
  'Dad‚Äôs Root Beer',
  'Don‚Äôt Be A Shit',
]

res5_menu_encodings = [
  'Caf‚Àö¬© Au Lait',
  'Dad‚Äö√Ñ√¥S Root Beer',
  'Don‚Äö√Ñ√¥T Be A Shit'
]

# '2HRX9P6HKXA8V'
res6_sales_encodings = [
  "Hans' Jalape√±O & Cheddar",
  'Hofbr√§U Dunkle',
  'Hofbr√§U Sommerzwickl',
  'Ros√©'
]

res6_menu_encodings = [
  "Hans' Jalape‚Àö¬±O & Cheddar",
  'Hofbr‚Àö¬Ssu Dunkle',
  'Hofbr‚Àö¬Ssu Sommerzwickl',
  'Ros‚Àö¬©',
]

'3AXDVZJYN9DRS'

In [ ]:
encodings_df1 = pd.DataFrame(zip(res1_menu_encodings, res1_sales_encodings))
encodings_df1 

'C0BE4NDSW26QN'

In [ ]:
encodings_df2 = pd.DataFrame(zip(res2_menu_encodings, res2_sales_encodings))
encodings_df2

'L3XS7WSJ4AJA3'

In [ ]:
encodings_df3 = pd.DataFrame(zip(res3_menu_encodings, res3_sales_encodings))
encodings_df3

'LBMCPAYT7W36V'

In [ ]:
encodings_df4 = pd.DataFrame(zip(res4_menu_encodings, res4_sales_encodings))
encodings_df4

'ED5J990H5VAZT'

In [ ]:
encodings_df5 = pd.DataFrame(zip(res5_menu_encodings, res5_sales_encodings))
encodings_df5

'2HRX9P6HKXA8V'

In [ ]:
encodings_df6 = pd.DataFrame(zip(res6_menu_encodings, res6_sales_encodings))
encodings_df6

Aggregate

In [ ]:
sales_to_remove = [res1_sales_to_remove, res5_sales_to_remove]
encodings_df = pd.concat([encodings_df1, encodings_df2, encodings_df3, encodings_df4, encodings_df5, encodings_df6])

Save to remove in cleaning file

In [ ]:
%store encodings_df 
%store sales_to_remove